### Import packages

In [26]:
from pathlib import Path
import json
import textdescriptives as td
from tqdm import tqdm
import pandas as pd
import numpy as np

### Define paths and data

In [27]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/raw"

fantasy_data_path = DATA_DIR / "generated_stories_fantasy.jsonl"
romance_data_path = DATA_DIR / "generated_stories_romance.jsonl"
litterary_fiction_data_path = DATA_DIR / "generated_stories_litterary_fiction.jsonl"

In [28]:
story_type = "romance"
DATA_DIR = PROJECT_ROOT / "data/processed"
OUTPUT_FILE = DATA_DIR / f"{story_type}_stories_text_descriptors.jsonl"

In [29]:
# Read JSONL file
with open(romance_data_path, 'r') as f:
    romance_stories = [json.loads(line) for line in f]

### Get text descriptive measures

In [30]:
# possible metrics
td.get_valid_metrics()

{'all',
 'coherence',
 'dependency_distance',
 'descriptive_stats',
 'information_theory',
 'pos_proportions',
 'quality',
 'readability'}

In [31]:
for item in tqdm(romance_stories, desc="Processing stories"):
    metrics = td.extract_metrics(
        text=item["story"],
        metrics=['all'],
        lang="en",
        spacy_model="en_core_web_lg"
    )

    cleaned_metrics = {}
    for key, value in metrics.items():
        if key == "text":
            continue
        # Convert Series/DataFrame cells to scalars
        if isinstance(value, pd.Series):
            cleaned_metrics[key] = value.iloc[0]
        else:
            cleaned_metrics[key] = value

    item.update(cleaned_metrics)

Processing stories:   0%|          | 0/6 [00:00<?, ?it/s]

ℹ Both a spacy model and a language were provided. Will use the spacy
model and ignore language.


Processing stories:  17%|█▋        | 1/6 [00:04<00:23,  4.79s/it]

ℹ Both a spacy model and a language were provided. Will use the spacy
model and ignore language.


Processing stories:  33%|███▎      | 2/6 [00:08<00:17,  4.38s/it]

...

Processing stories:  67%|██████▋   | 4/6 [00:16<00:08,  4.11s/it]

ℹ Both a spacy model and a language were provided. Will use the spacy
model and ignore language.


Processing stories:  83%|████████▎ | 5/6 [00:20<00:04,  4.04s/it]

ℹ Both a spacy model and a language were provided. Will use the spacy
model and ignore language.


Processing stories: 100%|██████████| 6/6 [00:24<00:00,  4.11s/it]


In [32]:
romance_stories

[{'story': "Beginning\n\nElena had cultivated precision the way other people cultivated gardens: with careful pruning, exact measurements, and a small, private ruthlessness. For ten years she had lived inside a stainless-steel kingdom—an open kitchen in a city that never truly slept, where the heat of the line was a constant and the applause for a tasting menu came as quickly as the next reservation. She could file a perfect brunoise of carrot with one glance, could coax a sauce into velvet silence with the same hand that had, once, typed the love letters she had sent and received in her twenties.\n\nWhen she finally booked a week on the island of Isola Verde, it was not bravado so much as exhaustion. The restaurant had been granted a new Michelin star and with it a roster of interviews and invitations and obligations Elena no longer possessed energy for. She wanted clay under her nails—something visceral, unmeasured. She wanted seeds to be seeds again instead of numbers on a procureme

In [33]:
# Pick the metrics you care about
keys_to_keep = [
    "story",
    "genre",
    "story_hint",
    "agent",
    "agent_type",
    "event",
    "event_valence",
    "context",
    "context_valence",
    "model",
    "date_of_generation",
    "output_tokens",
    "first_order_coherence", # how well each sentence connects to the previous one. Closer to 1 = smoother flow.
    "second_order_coherence", # coherence across larger spans (e.g., paragraphs). Higher values = more globally coherent text.
    "flesch_reading_ease", # readability - Higher is easier. 60–70 is standard for general reading; 80+ is very easy.
    "flesch_kincaid_grade", # readability - U.S. school grade level needed to read the text. 5.7 → ~6th grade.
    "dependency_distance_mean", # coherence/structure - Average “distance” between syntactically linked words. Higher = more complex sentences.
    "doc_length", # length
    "mean_word_length", # mean length of words; higher = more complex vocabulary.
    "proportion_unique_tokens", # Lexical diversity; higher = more varied vocabulary.
    "pos_prop_ADJ", # % of adjectives
    "pos_prop_NOUN", # % of nouns
    "pos_prop_VERB", # % of verbs
    "pos_prop_PRON", # % of pronouns
    "duplicate_ngram_chr_fraction_5", # repetitiveness - High value → story repeats the same phrases or chunks of text often
    "passed_quality_check", # quality
    "lix", # readability - Higher is more difficult. 20–30 is easy; 40–50 is difficult.
    "oov_ratio" # fraction of out-of-vocabulary words
]

In [34]:
with open(OUTPUT_FILE, "w") as f:
    for item in tqdm(romance_stories, desc="Saving JSONL"):
        # Keep only selected keys
        filtered_item = {k: item[k] for k in keys_to_keep if k in item}

        # Convert numpy types and round floats
        for k, v in filtered_item.items():
            if isinstance(v, (np.bool_, np.bool8)):
                filtered_item[k] = bool(v)
            elif isinstance(v, (np.integer, np.int32, np.int64)):
                filtered_item[k] = int(v)
            elif isinstance(v, (np.floating, np.float32, np.float64)):
                filtered_item[k] = round(float(v), 2)  # round floats to 2 decimals

        # Write as one JSON object per line
        f.write(json.dumps(filtered_item) + "\n")

Saving JSONL:   0%|          | 0/6 [00:00<?, ?it/s]/var/folders/wv/k1c_2q2x52q536wp2_p2kdpm0000gn/T/ipykernel_74696/3372902292.py:8: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if isinstance(v, (np.bool_, np.bool8)):
Saving JSONL: 100%|██████████| 6/6 [00:00<00:00, 5137.98it/s]
